 ## Imanol Mendoza Saenz de Buruaga
# Proyecto Avanzado de Procesamiento de Lenguaje Natural: Sistema RAG Multimodal para la Clasificación de Deficiencias Nutricionales en Cultivos

Este notebook implementa un flujo de trabajo (workflow) avanzado de **Generación Aumentada por Recuperación (RAG) Multimodal e Híbrida**. El objetivo principal es mejorar la precisión del diagnóstico de deficiencias de nutrientes en hojas de plantas, utilizando como caso de estudio el dataset de **Hojas de Plátano (Banana Deficiency)**.

---

## Arquitectura General del Workflow

El sistema se divide en cuatro fases críticas distribuidas de la siguiente manera:

1. **Configuración y Línea Base (Baseline):** Clonación del repositorio oficial de **AgriCLIP** y preparación del entorno de ejecución con soporte para aceleración por hardware (GPU).
2. **Construcción del Índice Vectorial Semántico:** Carga de la base de datos de conocimiento aumentado de los autores (archivo de captions enriquecidos), generación de embeddings de texto con el *Text Encoder* de **CLIP (ViT-B/32)** e indexación en la base de datos vectorial persistente **ChromaDB**.
3. **Canalización RAG Multimodal Híbrida:** - **Subtitulado Ciego:** Un LLM de visión analiza las imágenes *downstream* y genera una descripción puramente visual (colores, texturas, patrones) sin nombres de enfermedades ni tecnicismos biológicos para evitar sesgos.
   - **Fusión Temprana (Embedding Híbrido):** Se calcula el vector visual proyectado mediante el alineador lineal de AgriCLIP y se promedia con el embedding textual de la descripción visual.
   - **Recuperación e Inferencia:** Se realiza una búsqueda de los $K$ vecinos más cercanos ($K=3$) en ChromaDB empleando distancia coseno y se envía el contexto recuperado a `gpt-4o-mini` para que dicte un veredicto estructurado en formato JSON.
4. **Evaluación de Métricas:** Medición del rendimiento del clasificador (Accuracy), la calidad del recuperador (Recall@K, Precision@K, MRR) y la fidelidad del generador (Token F1, ROUGE-L, BERTScore).

---

## Prerrequisitos y Dependencias Críticas

Para ejecutar este notebook de forma exitosa, es estrictamente necesario contar con los siguientes elementos antes de iniciar:

### 1. Dataset de Preentrenamiento (Autores de AgriCLIP)
El índice vectorial se alimenta de las descripciones enriquecidas del conocimiento agrícola de AgriCLIP. Debes descargar el archivo correspondiente directamente desde el repositorio o la ruta compartida de los autores:
* **Enlace de descarga:** [AgriCLIP Pre-Training OneDrive](https://mbzuaiac-my.sharepoint.com/personal/umair_nawaz_mbzuai_ac_ae/_layouts/15/onedrive.aspx?id=%2Fpersonal%2Fumair%5Fnawaz%5Fmbzuai%5Fac%5Fae%2FDocuments%2FAgriCLIP%2FDataset%2FPre%2DTraining&ga=1) o buscar el enlace directamente del repo tomado como referencia en ALive Dataset Access : [Alive Dataset](https://github.com/umair1221/AgriCLIP)
* **Archivo requerido:** `AgriClip-Image-Prompts-Customized.csv` y subirlo a la raíz de este entorno de Google Colab.

### 2. Archivo de Evaluación Downstream
Se requiere el archivo `dataset_downstream_subtitulado.csv` en la raíz de Colab. Este archivo actúa como nuestro conjunto de prueba (test set) y debe contener las columnas con las rutas locales de las imágenes de banana que van a ser evaluadas a lo largo de la canalización.

### 3. Llave de API de OpenAI (Secretos de Colab)
El proceso de subtitulado ciego y la síntesis diagnóstica final dependen de los modelos fundacionales de OpenAI a través de llamadas de API por lotes.
* Debes generar una API Key válida desde tu consola de OpenAI.
* **Nota importante:** Asegúrate de contar con créditos disponibles o fondos suficientes en tu cuenta de OpenAI, ya que el procesamiento consume tokens de los modelos `gpt-4o-mini`.
* Configura la llave en el menú lateral izquierdo de Colab 🔑 (**Secretos / Userdata**) bajo el nombre exacto de: **`OpenAI`**.

In [1]:
# Instalación de dependencias del sistema y modelos
!pip install google-genai
!pip install git+https://github.com/openai/CLIP.git
!pip install timm open_clip_torch ftfy regex tqdm evaluation pandas setuptools
!pip install chromadb
!pip install rouge-score bert-score scikit-learn
!pip install openai

# Sistema operativo, control de archivos y tiempo
import os
import sys
import shutil
import time

# Manipulación de datos y descarga de datasets
import pandas as pd
import numpy as np
import kagglehub

# Aprendizaje profundo y procesamiento de imágenes y CLIP
import torch
import torchvision.transforms as transforms
from PIL import Image
import clip

# APIs de IA generativa y credenciales seguras
from openai import OpenAI
from google import genai
from google.colab import userdata

# Almacenamiento vectorial, formatos de datos y utilidades
import chromadb
import json
import base64
import random
from tqdm import tqdm

# Métricas de evaluación, NLP y aprendizaje estadístico
from sklearn.metrics import accuracy_score
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-bg5cx23u
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-bg5cx23u
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=96143cd44b7be77991751adc902f9803071340b9179dcfb5e17d87f0c3f33c3a
  Stored in directory: /tmp/pip-ephem-wheel-cache-1ep1g4it/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 89.3 MB/s eta 0:00:00
  Created wheel for evaluation: filename=evaluation-0.0

## Fase 1: Configuración del Entorno y Línea Base (Baseline)

En esta primera fase se prepara la infraestructura de software necesaria para garantizar la reproducibilidad del proyecto y se configura el conjunto de datos de prueba para establecer un punto de comparación directo con los resultados del artículo científico original.

### 1.1 Instalación de Dependencias Críticas
El pipeline híbrido requiere la coexistencia de múltiples librerías de visión por computadora, procesamiento de lenguaje natural y almacenamiento vectorial. En este primer bloque se instalan de forma automatizada:
* **`google-genai` y `openai`:** SDKs oficiales para interactuar con los modelos fundacionales que realizarán el subtitulado descriptivo y la síntesis diagnóstica.
* **CLIP de OpenAI:** Clonado e instalado directamente desde su repositorio oficial en GitHub para disponer del codificador de texto original (`ViT-B/32`).
* **Librerías de Soporte Multimodal:** `timm`, `open_clip_torch`, `regex`, `ftfy` y herramientas de evaluación para la manipulación y normalización de las arquitecturas de red.
* **ChromaDB:** Motor de base de datos vectorial embebido y persistente donde se indexará el conocimiento agrícola.

### 1.2 Clonación de AgriCLIP y Configuración de Rutas
Se realiza la clonación del repositorio oficial de los autores (`https://github.com/umair1221/AgriCLIP.git`) y se manipula dinámicamente la variable de entorno `PYTHONPATH` del sistema. Esto permite inyectar el directorio local `/content/AgriCLIP` en el entorno de Python para poder importar de manera nativa los módulos propietarios desarrollados por los investigadores, tales como la clase `TextToConcept`.

### 1.3 Descarga y Estructuración del Dataset Downstream (Banana Deficiency)
Utilizando la herramienta `kagglehub`, se automatiza la descarga del dataset de evaluación referenciado en el artículo: **Nutrient Deficient Banana Plant Leaves**.
* El script extrae los archivos desde el caché global de Kaggle y los transfiere a la ruta estructurada interna del proyecto: `./Dataset/downstream_data/Banana Deficiency`.
* Esta organización en espejo simula exactamente el entorno de almacenamiento que el código fuente de AgriCLIP espera para mapear las imágenes crudas de las hojas de plátano infectadas o deficientes.

### 1.4 Configuración del Script de Inferencia Zero-Shot (Línea Base)
Finalmente, la ejecución para el script `AgriClip_zeroshot.py`. Este bloque representa el **Baseline** del proyecto. Utiliza los pesos preentrenados de Dino (`dino_pretrain.pth`) y el alineador lineal entrenado por los autores (`Agri_Dino_aligner_DPT_CPT.pth`) para clasificar de forma tradicional (*zero-shot*) las hojas de plátano, sirviendo como la métrica de control contra la cual mediremos la ganancia de nuestro sistema RAG multimodal.

In [2]:
# Clonar el repositorio oficial de AgriCLIP
!git clone https://github.com/umair1221/AgriCLIP.git

# 2. Movernos al directorio del proyecto
%cd AgriCLIP
os.environ['PYTHONPATH'] = f"./:{os.environ.get('PYTHONPATH', '')}"
sys.path.append(os.getcwd())

print("¡Repositorio clonado y entorno preparado con éxito!")

Cloning into 'AgriCLIP'...
remote: Enumerating objects: 447, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 447 (delta 111), reused 91 (delta 35), pack-reused 256 (from 2)
Receiving objects: 100% (447/447), 113.92 MiB | 19.55 MiB/s, done.
Resolving deltas: 100% (196/196), done.
/content/AgriCLIP
¡Repositorio clonado y entorno preparado con éxito!


### Dataset

Cargamos de kaggle el data set que usaremos para evaluar nuestra metodologia y de baseline, se encuentra en las referencias del artículo: https://www.kaggle.com/datasets/warcoder/nutrient-deficient-banana-plant-leaves/data?select=Nutrient+Deficient+Banana+Plant+Leaves

In [3]:

path_descarga = kagglehub.dataset_download("warcoder/nutrient-deficient-banana-plant-leaves")
print(f"Descargado en la ruta temporal: {path_descarga}")

destino_banana = "/content/AgriCLIP/Dataset/downstream_data/Banana Deficiency"
os.makedirs(destino_banana, exist_ok=True)

if os.path.exists(path_descarga):
    for item in os.listdir(path_descarga):
        origen = os.path.join(path_descarga, item)
        destino = os.path.join(destino_banana, item)

        if os.path.isdir(origen):
            if os.path.exists(destino):
                shutil.rmtree(destino)
            shutil.copytree(origen, destino)
        else:
            shutil.copy2(origen, destino)
    print(f"¡Dataset de Banana estructurado con éxito en: {destino_banana}!")
else:
    print("Error: No se encontró la ruta de origen de kagglehub.")

100%|██████████| 157M/157M [00:09<00:00, 16.9MB/s]

Extracting files...


Descargado en la ruta temporal: /root/.cache/kagglehub/datasets/warcoder/nutrient-deficient-banana-plant-leaves/versions/1
¡Dataset de Banana estructurado con éxito en: /content/AgriCLIP/Dataset/downstream_data/Banana Deficiency!


### Creación de nuestro baseline

In [6]:
%cd /content/AgriCLIP

!python AgriCLIP_alignment/AgriClip_zeroshot.py \
    --dataset-name "Banana Deficiency" \
    --data-path "./Dataset/downstream_data/Banana Deficiency/Nutrient Deficient Banana Plant Leaves/Nutrient Deficient RAW Images of Banana Leaves" \
    --dino-path "Weights/dino_pretrain.pth" \
    --aligner-path "Weights/Aligned_Models/Agri_Dino_aligner_DPT_CPT.pth" \
    --batch-size 32 \
    --num-workers 2

/content/AgriCLIP
Using device: cpu
Downloading: "https://github.com/facebookresearch/dino/zipball/main" to /root/.cache/torch/hub/main.zip
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Downloading: "https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/dino_resnet50_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dino_resnet50_pretrain.pth
100% 90.0M/90.0M [00:01<00:00, 54.0MB/s]
100%|████████████████████████████████████████| 335M/335M [00:02<00:00, 133MiB/s]
['boron', 'calcium', 'healthy', 'iron', 'magnesium', 'manganese

## Fase 2: Construcción del Índice Vectorial Semántico (ChromaDB)

En esta fase se procesa el corpus de conocimiento agrícola provisto por los autores de AgriCLIP y se construye un almacén vectorial persistente. El objetivo es indexar semánticamente los fragmentos de texto descriptivos (*captions*) para que puedan ser recuperados de forma eficiente durante la ejecución del pipeline RAG.

### 2.1 Carga del Corpus y Manejo de Líneas Corruptas
El archivo original `AgriClip-Image-Prompts-Customized.csv` puede contener inconsistencias en su estructura debido al uso de comas internas dentro de los textos descriptivos, lo que rompe el parseo tradicional de un archivo CSV.
* Para resolver esto, se implementa una rutina de lectura robusta utilizando `open()` línea por línea.
* Se descartan los registros defectuosos que no cumplen con la estructura exacta esperada, aislando de forma limpia las columnas críticas: `image_path` y `caption`.

### 2.2 Inicialización de ChromaDB Persistente
Se configura un cliente de **ChromaDB** apuntando a un directorio local en el entorno de ejecución (`./chroma_db`). Esto asegura que la base de datos se almacene de forma física y persistente, evitando la necesidad de recalcular los embeddings si se detiene el backend del notebook.
* Se crea (o recupera, si ya existe) una colección específica llamada **`agriclip_collection`**.
* Se configura la métrica de distancia de la colección como **Distancia Coseno (`cosine`)**, la cual es ideal para medir la similitud semántica entre vectores normalizados de texto e imagen.

### 2.3 Generación de Embeddings con el Codificador de Texto de CLIP
En lugar de utilizar embeddings genéricos, la colección se alimenta directamente con los vectores generados por el codificador de texto de **CLIP (`ViT-B/32`)**, preservando la alineación semántica del espacio multimodal original.
* **Procesamiento por Lotes (Batching):** Las descripciones textuales se procesan en bloques de tamaño fijo ($N=64$) para optimizar el uso de memoria en la GPU.
* **Tokenización y Extracción:** Cada bloque de texto se tokeniza usando `clip.tokenize()`, se procesa a través de las capas transformadoras del modelo (`model.encode_text`) y se normaliza matemáticamente mediante su norma L2.

### 2.4 Almacenamiento e Indexación de Datos
Finalmente, los embeddings de texto generados son insertados en la base de datos vectorial acompañados de sus respectivos identificadores únicos (`ids`) y metadatos complementarios (`metadata`), los cuales contienen la ruta física de la imagen original a la que pertenece cada descripción. Una vez finalizado el ciclo, la base de datos queda indexada y lista para responder a consultas semánticas.

In [4]:
os.makedirs('/content/AgriCLIP/AgriCLIP_RAG/vector_db', exist_ok=True)

### Base de datos utilizada por ellos

Base de datos con ruta de la imagen y su descripción aumentada (descargar archivo customized y poner en archivos de colab): https://mbzuaiac-my.sharepoint.com/personal/umair_nawaz_mbzuai_ac_ae/_layouts/15/onedrive.aspx?id=%2Fpersonal%2Fumair%5Fnawaz%5Fmbzuai%5Fac%5Fae%2FDocuments%2FAgriCLIP%2FDataset%2FPre%2DTraining&ga=1

In [5]:
ruta_origen = "/content/AgriClip-Image-Prompts-Customized.csv"
carpeta_destino = "/content/AgriCLIP/Dataset/downstream_data"
ruta_destino = os.path.join(carpeta_destino, "AgriClip-Image-Prompts-Customized.csv")

os.makedirs(carpeta_destino, exist_ok=True)

if os.path.exists(ruta_origen):
    shutil.move(ruta_origen, ruta_destino)
    print(f"¡Archivo acomodado con éxito en: {ruta_destino}!")
else:
    if os.path.exists(ruta_destino):
        print(f"El archivo ya se encuentra en la ruta correcta: {ruta_destino}")
    else:
        print("🚨 Error: No encontré el archivo en la raíz de Colab. Asegúrate de haberlo subido con ese nombre exacto.")

¡Archivo acomodado con éxito en: /content/AgriCLIP/Dataset/downstream_data/AgriClip-Image-Prompts-Customized.csv!


El archivo viene en un formato con muchos carácteres entonces su carga tiene que ser cuidadosa

In [6]:
def load_corrupted_csv(filepath):
    data = []

    with open(filepath, 'r', encoding='latin1') as f:
        lines = f.readlines()

    # Empezamos desde la línea 1 para saltar el encabezado (filepath,caption)
    for line in lines[1:]:
        line = line.strip()
        if not line:
            continue

        # Quitar las comillas globales que envuelven todo el renglón
        if line.startswith('"'): line = line[1:]
        if line.endswith('"'): line = line[:-1]

        # Quitar las comas basura al final
        line = line.rstrip(',')

        # Separar por la primera coma que divide el path del caption
        parts = line.split(',', 1)

        if len(parts) == 2:
            path = parts[0].strip()
            caption = parts[1].strip()

            # Limpiar las comillas dobles extra en el texto del caption (""texto"")
            caption = caption.strip('"')

            data.append({"filepath": path, "caption": caption})

    df = pd.DataFrame(data)
    return df

In [7]:
csv_path = "/content/AgriCLIP/Dataset/downstream_data/AgriClip-Image-Prompts-Customized.csv"
df = load_corrupted_csv(csv_path)
df.tail(10)

,filepath,caption
413505,/home/umair.nawaz/Research_Work/Main-DATA/My_S...,Tomato leaf exhibiting Late Blight symptoms wi...
413506,/home/umair.nawaz/Research_Work/Main-DATA/My_S...,"Late Blight disease on a tomato leaf, showing ..."
413507,/home/umair.nawaz/Research_Work/Main-DATA/My_S...,A tomato leaf severely affected by Late Blight...
413508,/home/umair.nawaz/Research_Work/Main-DATA/My_S...,"Late Blight disease on a tomato leaf, showing ..."
413509,/home/umair.nawaz/Research_Work/Main-DATA/My_S...,"Signs of Late Blight on a tomato leaf, charact..."
413510,/home/umair.nawaz/Research_Work/Main-DATA/My_S...,"Signs of Late Blight on a tomato leaf, charact..."
413511,/home/umair.nawaz/Research_Work/Main-DATA/My_S...,"Signs of Late Blight on a tomato leaf, charact..."
413512,/home/umair.nawaz/Research_Work/Main-DATA/My_S...,"Late Blight disease on a tomato leaf, showing ..."
413513,/home/umair.nawaz/Research_Work/Main-DATA/My_S...,A tomato leaf severely affected by Late Blight...
413514,/home/umair.nawaz/Research_Work/Main-DATA/My_S...,"Signs of Late Blight on a tomato leaf, charact..."


In [8]:

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Usando dispositivo: {device}")
    if device == "cpu":
        print("🚨 ALERTA: Estás usando CPU. El proceso será muy lento. Cambia a GPU T4 en Entorno de Ejecución.")

    print("Cargando modelo CLIP...")
    model, preprocess = clip.load("ViT-B/32", device=device)

    chroma_client = chromadb.PersistentClient(path="/content/AgriCLIP/AgriCLIP_RAG/vector_db")
    collection = chroma_client.get_or_create_collection(
        name="alive_knowledge_base",
        metadata={"hnsw:space": "cosine"}
    )

    #  Cargar archivo con nuestro parseador manual
    csv_path = "/content/AgriCLIP/Dataset/downstream_data/AgriClip-Image-Prompts-Customized.csv"
    df = load_corrupted_csv(csv_path)

    batch_size = 128
    total_rows = len(df)
    print(f"Total de registros perfectamente recuperados: {total_rows}")

    if total_rows == 0:
        print("Error: Aún no se recuperan datos. Verifica que el archivo no esté vacío.")
        return

    # Bucle de indexación por lotes
    for i in tqdm(range(0, total_rows, batch_size), desc="Indexando ALive en ChromaDB"):
        batch_df = df.iloc[i:i+batch_size]

        captions = batch_df['caption'].tolist()

        # Generar tokens y embeddings
        try:
            text_tokens = clip.tokenize(captions, truncate=True).to(device)
            with torch.no_grad():
                text_features = model.encode_text(text_tokens)
                text_features /= text_features.norm(dim=-1, keepdim=True)
                embeddings = text_features.cpu().numpy().tolist()
        except Exception as e:
            print(f"\nSaltando lote en el índice {i}: {e}")
            continue

        metadatas = []
        documents = []
        ids = []

        for idx, row in batch_df.iterrows():
            f_path = str(row['filepath'])
            parts = f_path.split('/')

            domain = "Unknown"
            specie_class = "Unknown"

            if "Fish" in parts or "Fisheries" in parts: domain = "Fisheries"
            elif "Crops" in parts: domain = "Crops"
            elif "Livestock" in parts: domain = "Livestock"

            if len(parts) > 2:
                specie_class = parts[-2]

            metadatas.append({
                "domain": domain,
                "specie_or_class": specie_class,
                "original_path": f_path
            })
            documents.append(str(row['caption']))
            ids.append(f"alive_{idx}")

        collection.add(
            embeddings=embeddings,
            documents=documents,
            metadatas=metadatas,
            ids=ids
        )

    print(f"\n¡Fase 2 Completada! ChromaDB tiene ahora {collection.count()} registros.")

if __name__ == "__main__":
    main()

Usando dispositivo: cuda
Cargando modelo CLIP...


100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 119MiB/s]


Total de registros perfectamente recuperados: 413515


Indexando ALive en ChromaDB: 100%|██████████| 3231/3231 [16:17<00:00,  3.31it/s]



¡Fase 2 Completada! ChromaDB tiene ahora 413515 registros.


## Fase 3: Canalización RAG Multimodal Híbrida e Inferencia

Esta fase constituye el núcleo del proyecto. Se implementa un pipeline híbrido diseñado específicamente para evitar el *Data Leakage* (filtración de datos). En lugar de clasificar la imagen directamente con etiquetas preestablecidas, el sistema extrae las características visuales puras de la muestra, recupera conocimiento experto relacionado y delega una inferencia razonada a un Modelo de Lenguaje Grande (LLM).

El flujo de trabajo por cada muestra del dataset de prueba (`dataset_downstream_subtitulado.csv`) se compone de los siguientes pasos analíticos:

### 3.1 Subtitulado Ciego (Blind Captioning) con GPT-4o-mini
Para romper el sesgo de clasificación directa, la imagen de la hoja de plátano se envía primero a `gpt-4o-mini` (utilizando capacidades de visión) bajo un *system prompt* sumamente estricto:
* **Restricción:** El modelo tiene estrictamente prohibido mencionar nombres de enfermedades, patógenos o deficiencias nutricionales (como "Sigatoka", "Potasio", "Cordana", etc.).
* **Objetivo:** Generar una descripción morfológica puramente visual y anatómica de la hoja, detallando patrones de color (clorosis, necrosis), formas de las manchas, bordes, distribución de las lesiones y texturas.

### 3.2 Fusión Temprana de Embeddings (Embedding Híbrido)
Una vez que se obtiene el subtítulo ciego, se construye una representación vectorial híbrida que combina el espacio visual y el espacio textual:
1. **Embedding Textual ($E_{text}$):** Se calcula el vector del subtítulo ciego utilizando el codificador de texto de CLIP.
2. **Embedding Visual ($E_{visual}$):** Se extraen los mapas de características de la imagen mediante la red espinal de AgriCLIP (Dino ResNet50) y se proyectan al espacio alineado mediante el módulo `TextToConcept` preentrenado por los autores.
3. **Fusión por Promedio:** Ambos vectores se promedian matemáticamente y se normalizan utilizando la norma L2, generando un **embedding híbrido unificado** que captura tanto la señal sensorial cruda como la abstracción lingüística de la muestra.

### 3.3 Recuperación de Contexto Experto (Retrieval)
El embedding híbrido unificado se utiliza como query para realizar una búsqueda de los **$K$ vecinos más cercanos ($K=3$)** dentro de la colección de ChromaDB configurada en la Fase 2.
* El sistema calcula la distancia coseno contra todo el corpus de conocimiento indexado.
* Retorna los 3 fragmentos de texto (*captions* enriquecidos de los autores) que guardan la mayor similitud semántica y morfológica con la muestra analizada, sirviendo como el "contexto de soporte médico-agrícola".

### 3.4 Inferencia y Síntesis Diagnóstica Orientada a Objetos
Finalmente, se construye un prompt estructurado que fusiona el **subtítulo ciego** (lo que se ve en la hoja) con el **contexto recuperado de ChromaDB** (el conocimiento experto de soporte).
* Este prompt se envía a un pipeline de generación que exige una respuesta en un formato estrictamente estructurado en **JSON** empleando la API de OpenAI.
* El LLM actúa como un fitopatólogo experto que correlaciona la sintomatología visual con la literatura científica recuperada para emitir un veredicto final. El JSON resultante incluye campos clave como: `predicted_label` (diagnóstico final), `confidence` (nivel de confianza) y `reasoning` (justificación del diagnóstico).

**Requerimientos:** subir archivo "dataset_downstream_subtitulado", contiene ruta de imagen,imagen_id y etiqueta real

In [9]:
def encode_image_to_base64(image_path):
    """Convierte una imagen local en una cadena Base64 para enviarla a la API."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def generar_descripcion_planta(client, base64_image, prompt_estandar):
    """Llama a gpt-4o-mini pasando la imagen y el prompt sin sesgos."""
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt_estandar},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            temperature=0.2,
            max_tokens=50
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"\nError al procesar una imagen con la API de OpenAI: {e}")
        return "Error_Generacion"

def main():
    print("=== Iniciando Canalización de Visión por Computadora con OpenAI ===")

    try:
        api_key_openai = userdata.get('OpenAI')
        client = OpenAI(api_key=api_key_openai)
        print("Cliente OpenAI inicializado con éxito desde tus secretos.")
    except Exception as e:
        print(" Error Crítico: No se encontró el secreto 'OpenAI' en tu entorno de Colab.", e)
        return

    ruta_csv_origen = "/content/dataset_downstream_subtitulado.csv"
    ruta_csv_destino = "/content/dataset_downstream_subtitulado_PROCESADO.csv"

    if not os.path.exists(ruta_csv_origen):
        print(f"Error: No se encontró el archivo base en {ruta_csv_origen}. Asegúrate de subirlo primero.")
        return

    df = pd.read_csv(ruta_csv_origen)
    print(f"Archivo cargado correctamente. Total de registros a procesar: {len(df)}")

    prompt_estandar = (
        "Act as a purely descriptive and objective visual observer. "
        "Describe only what is visible in the plant leaf: exact colors, textures, shapes, "
        "spot patterns, or geometric deformations.\n"
        "STRICT RULES:\n"
        "1. Do not diagnose or mention any nutrient deficiency, disease, or fungus name.\n"
        "2. Do not use words like 'symptom', 'healthy', 'pathology', or 'damage'.\n"
        "3. Do not use scientific or biological jargon (e.g., 'chlorophyll', 'interveinal chlorosis', 'necrosis'). "
        "Instead, use simple descriptions like 'yellow spots between veins' or 'brown dry edges'.\n"
        "OUTPUT FORMAT: Generate a single description in English. Do not exceed 25 words."
    )

    descripciones_generadas = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Procesando imágenes"):
        ruta_img = row['ruta_imagen']

        if not os.path.exists(ruta_img):
            print(f"\nAdvertencia: La imagen en {ruta_img} no existe. Saltando registro.")
            descripciones_generadas.append("Imagen_No_Encontrada")
            continue

        base64_img = encode_image_to_base64(ruta_img)
        descripcion = generar_descripcion_planta(client, base64_img, prompt_estandar)

        descripciones_generadas.append(descripcion)

        time.sleep(1)

    df['descripcion_vision_llm'] = descripciones_generadas
    df.to_csv(ruta_csv_destino, index=False, encoding='utf-8')

    print("\n" + "="*60)
    print("¡PROCESO DE SUBTITULADO COMPLETADO AL 100%! 🏆")
    print(f"El nuevo archivo sin sesgos se guardó en: {ruta_csv_destino}")
    print("="*60)

if __name__ == "__main__":
    main()

=== Iniciando Canalización de Visión por Computadora con OpenAI ===
Cliente OpenAI inicializado con éxito desde tus secretos.
Archivo cargado correctamente. Total de registros a procesar: 90


Procesando imágenes: 100%|██████████| 90/90 [03:50<00:00,  2.56s/it]


¡PROCESO DE SUBTITULADO COMPLETADO AL 100%! 🏆
El nuevo archivo sin sesgos se guardó en: /content/dataset_downstream_subtitulado_PROCESADO.csv


In [10]:

%cd /content/AgriCLIP
if "/content/AgriCLIP" not in sys.path:
    sys.path.insert(0, "/content/AgriCLIP")
if "/content/AgriCLIP/AgriCLIP_alignment" not in sys.path:
    sys.path.insert(0, "/content/AgriCLIP/AgriCLIP_alignment")

from TextToConcept import TextToConcept

def consultar_openai_estructurado(client, contexto_rag, clases_disponibles):
    """Modulo de Sintesis con formato estructurado nativo JSON usando OpenAI, 20 palabras y prioridad diagnostica"""

    prompt_sistema = (
        "You are an automated agricultural classification script. You must output a JSON object containing "
        "exactly two keys: 'prediccion' and 'descripcion'. No conversational text or markdown allowed."
    )

    prompt_usuario = f"""
    Analyze the following RAG context strings from a plant database.

    CONTEXT FROM CHROMADB:
    {contexto_rag}

    ALLOWED TARGET CLASSES: {clases_disponibles}

    CRITICAL CLASSIFICATION HIERARCHY AND PRIORITIZATION:
    1. ULTIMATE LAST RESORT RULE: Treating a plant as "healthy" must be your absolute last resort. If there is even a slight doubt, suspicion, or mention of any abnormality, you are strictly forbidden from choosing "healthy". Give maximum priority to classifying an anomaly.
    2. FORCE MAP STRATEGY: If the context mentions any symptoms, lesions, spots, or deficiencies (even unlisted ones like phosphorus or scab), you MUST force a mapping to the closest available nutrient deficiency from the ALLOWED TARGET CLASSES list using these visual translation rules:
       - If text describes veins, dark veins, yellowing between veins, or interveinal patterns -> choose "manganese" or "magnesium" or "iron"
       - If text describes edges, margins, browning, necrosis, or burnt edges -> choose "potassium"
       - If text describes wrinkles, wavy margins, crinkling, or distorted young leaves -> choose "calcium"
       - If text describes coiled, hook-like shape, or tightly rolled leaves -> choose "boron"
       - If it mentions general pale yellowish-green chlorosis -> choose "sulphur" or "zinc"
       - If a deficiency is present but ambiguous, pick "manganese" or "potassium" as the default deficiency candidate. NEVER output "healthy" if the context is not completely pristine.
    3. EXCLUSIVE HEALTHY CRITERIA: You can only output "healthy" if all text fragments unanimously and explicitly state that the leaf is perfectly healthy, vibrant, clean, and completely free of any damage or stress.

    OUTPUT FORMAT CONSTRAINTS:
    Your response must be a JSON object matching this schema. The 'descripcion' field MUST be an expert justification of maximum 20 words.
    {{
        "prediccion": "The chosen exact class name from the ALLOWED TARGET CLASSES list",
        "descripcion": "Expert concise justification using maximum 20 words"
    }}
    """

    for intento in range(5):
        try:
            response = client.chat.completions.create(
                model='gpt-4o-mini',
                messages=[
                    {"role": "system", "content": prompt_sistema},
                    {"role": "user", "content": prompt_usuario}
                ],
                temperature=0.0,
                response_format={"type": "json_object"}
            )
            res_json = json.loads(response.choices[0].message.content.strip())

            palabras = res_json.get("descripcion", "").split()
            if len(palabras) > 20:
                res_json["descripcion"] = " ".join(palabras[:20])

            return res_json
        except Exception as e:
            error_str = str(e)
            if "429" in error_str or "rate_limit" in error_str:
                print(f"\nRate limit reached. Cooling down API for 20 seconds (Attempt {intento+1}/5)...")
                time.sleep(20)
                continue
            else:
                return {"prediccion": "manganese", "descripcion": "Error parsing JSON"}

    return {"prediccion": "manganese", "descripcion": "Quota error limit"}

def main():
    print("=== Iniciando Pipeline RAG Hibrido Avanzado ===")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Dispositivo de computo detectado: {device}")

    # Inicializacion de OpenAI
    try:
        api_key_openai = userdata.get('OpenAI')
        ai_client = OpenAI(api_key=api_key_openai)
        print("Cliente OpenAI inicializado con exito.")
    except Exception as e:
        print("Error al recuperar el token 'OpenAI' de los Secretos de Colab:", e)
        return

    # Carga del archivo sin sesgos
    ruta_csv_maestro = "/content/dataset_downstream_subtitulado_PROCESADO.csv"
    if not os.path.exists(ruta_csv_maestro):
        print(f"Error Critico: El archivo maestro {ruta_csv_maestro} no existe.")
        return

    df_maestro = pd.read_csv(ruta_csv_maestro)
    clases_disponibles = list(df_maestro['folder_real'].unique())
    print(f"Archivo maestro cargado con {len(df_maestro)} filas.")
    print(f"Clases objetivo en el experimento: {clases_disponibles}")

    # Inicializacion del Modelo de Vision Dino ResNet50
    print("\n[Rama Visual] Inicializando Dino ResNet50...")
    dino_path = "Weights/dino_pretrain.pth"
    aligner_path = "Weights/Aligned_Models/Agri_Dino_aligner_DPT_CPT.pth"

    base_model = torch.hub.load('facebookresearch/dino:main', 'dino_resnet50', pretrained=True)
    base_model.load_state_dict(torch.load(dino_path, map_location=device, weights_only=True))

    def forward_features(x):
        features = base_model(x)
        return features.squeeze()
    base_model.forward_features = forward_features
    base_model.to(device)
    base_model.eval()

    text_to_concept_instance = TextToConcept(base_model, 'Dino')
    text_to_concept_instance.load_linear_aligner(aligner_path)

    transform_dino = transforms.Compose([
        transforms.Resize(224), transforms.CenterCrop(224), transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Inicializacion del Text Encoder de CLIP
    print("\n[Rama Textual] Inicializando Text Encoder de CLIP (ViT-B/32)...")
    clip_model, _ = clip.load("ViT-B/32", device=device)
    clip_model.eval()

    # Conexion a la Base de Datos Vectorial ChromaDB
    print("\n[Memoria Vectorial] Conectando a la base de datos persistente...")
    chroma_client = chromadb.PersistentClient(path="./AgriCLIP_RAG/vector_db")
    collection = chroma_client.get_collection(name="alive_knowledge_base")

    K_VECINOS = 3
    registros_evaluacion = []
    ruta_salida_resultados = "/content/resultados_canalizacion_rag.csv"

    print("\n" + "="*50)
    print("PROCESANDO VECTORES FUSIONADOS E INFERENCIA CIEGA CON GPT-4O-MINI")
    print("="*50)

    for idx, row in tqdm(df_maestro.iterrows(), total=len(df_maestro), desc="Matriz RAG"):
        ruta_completa_img = row['ruta_imagen']
        img_name = row['imagen_id']
        folder_real = row['folder_real']
        descripcion_visual = row['descripcion_vision_llm']

        try:
            img_input = Image.open(ruta_completa_img).convert("RGB")
            img_tensor = transform_dino(img_input).unsqueeze(0).to(device)
        except Exception as e:
            print(f"\nAdvertencia: Salto en {img_name}: {e}")
            continue

        #  Vectorizacion Visual (AgriCLIP)
        with torch.no_grad():
            features_dino = base_model.forward_features(img_tensor)
            if features_dino.dim() == 1:
                features_dino = features_dino.unsqueeze(0)

            W = text_to_concept_instance.linear_aligner.W
            b = text_to_concept_instance.linear_aligner.b
            W_corregida = W.t() if W.shape[0] != features_dino.shape[1] else W

            projected_visual_features = torch.matmul(features_dino, W_corregida) + b
            projected_visual_features /= projected_visual_features.norm(dim=-1, keepdim=True)

        # Vectorizacion Semantica
        with torch.no_grad():
            text_tokens = clip.tokenize([str(descripcion_visual)]).to(device)
            text_features = clip_model.encode_text(text_tokens)
            text_features /= text_features.norm(dim=-1, keepdim=True)

        #  Fusion Temprana (Embedding Hibrido)
        with torch.no_grad():
            fused_features = (projected_visual_features + text_features) / 2.0
            fused_features /= fused_features.norm(dim=-1, keepdim=True)
            vector_consulta_hibrido = fused_features.cpu().numpy().tolist()[0]

        # Recuperación
        resultados_chroma = collection.query(
            query_embeddings=[vector_consulta_hibrido],
            n_results=K_VECINOS
        )

        lista_vecinos = ["", "", ""]
        contexto_rag_texto_completo = ""

        for rank in range(len(resultados_chroma['ids'][0])):
            doc_recuperado = resultados_chroma['documents'][0][rank]
            meta_recuperado = resultados_chroma['metadatas'][0][rank]
            clase_origen = meta_recuperado.get('specie_or_class', 'Unknown')

            lista_vecinos[rank] = str(doc_recuperado)
            contexto_rag_texto_completo += f"Vecino {rank+1} [Origen: {clase_origen}]: {doc_recuperado}\n"

        #  Inferencia Estructurada con OpenAI (gpt-4o-mini) ---
        respuesta_json = consultar_openai_estructurado(ai_client, contexto_rag_texto_completo, clases_disponibles)

        clase_predicha_llm = respuesta_json.get("prediccion", "Unknown")
        descripcion_generada = respuesta_json.get("descripcion", "")

        registros_evaluacion.append({
            "imagen_id": img_name,
            "folder_real": folder_real,
            "prediccion_llm": clase_predicha_llm,
            "descripcion_sintomas_censurada": descripcion_visual,
            "vecino_1": lista_vecinos[0],
            "vecino_2": lista_vecinos[1],
            "vecino_3": lista_vecinos[2],
            "metadata_vecino_1": resultados_chroma['metadatas'][0][0].get('specie_or_class', 'Unknown') if len(resultados_chroma['metadatas'][0]) > 0 else "Unknown",
            "metadata_vecino_2": resultados_chroma['metadatas'][0][1].get('specie_or_class', 'Unknown') if len(resultados_chroma['metadatas'][0]) > 1 else "Unknown",
            "metadata_vecino_3": resultados_chroma['metadatas'][0][2].get('specie_or_class', 'Unknown') if len(resultados_chroma['metadatas'][0]) > 2 else "Unknown",
            "justificacion_final_llm": descripcion_generada
        })

        time.sleep(1)

        # Guardado incremental
        pd.DataFrame(registros_evaluacion).to_csv(ruta_salida_resultados, index=False, encoding='utf-8')

    print("\n" + "="*60)
    print("FASE 3 MULTIMODAL HIBRIDA COMPLETADA AL 100%")
    print(f"Los diagnosticos se consolidaron correctamente en: {ruta_salida_resultados}")
    print("="*60)

if __name__ == "__main__":
    main()

/content/AgriCLIP
=== Iniciando Pipeline RAG Hibrido Avanzado ===
Dispositivo de computo detectado: cuda
Cliente OpenAI inicializado con exito.
Archivo maestro cargado con 90 filas.
Clases objetivo en el experimento: ['manganese', 'potassium', 'calcium', 'zinc', 'healthy', 'sulphur', 'iron', 'boron', 'magnesium']

[Rama Visual] Inicializando Dino ResNet50...
Downloading: "https://github.com/facebookresearch/dino/zipball/main" to /root/.cache/torch/hub/main.zip


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Downloading: "https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/dino_resnet50_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dino_resnet50_pretrain.pth


100%|██████████| 90.0M/90.0M [00:00<00:00, 393MB/s]
100%|███████████████████████████████████████| 335M/335M [00:04<00:00, 82.3MiB/s]



[Rama Textual] Inicializando Text Encoder de CLIP (ViT-B/32)...

[Memoria Vectorial] Conectando a la base de datos persistente...

PROCESANDO VECTORES FUSIONADOS E INFERENCIA CIEGA CON GPT-4O-MINI


Matriz RAG: 100%|██████████| 90/90 [03:16<00:00,  2.19s/it]


FASE 3 MULTIMODAL HIBRIDA COMPLETADA AL 100%
Los diagnosticos se consolidaron correctamente en: /content/resultados_canalizacion_rag.csv


## Fase 4: Evaluación de Métricas y Resultados del Sistema

En esta fase final se cuantifica de forma rigurosa el rendimiento de cada uno de los componentes de la canalización RAG híbrida. A diferencia de un clasificador tradicional, un sistema RAG multimodal requiere una evaluación desacoplada en tres dimensiones críticas: la precisión del clasificador final, la calidad del motor de recuperación y la fidelidad del texto generado.

### 4.1 Métricas de Clasificación (vs. Baseline del Paper)
Se calcula el **Accuracy (Exactitud)** global del sistema comparando la etiqueta real del dataset (`folder_real`) contra la predicción final dictada por el LLM estructurado (`prediccion_llm`).
* Esta métrica nos permite contrastar directamente nuestro enfoque RAG híbrido y ciego contra la línea base de los autores (quienes reportan un **23.55%** de Accuracy utilizando AgriCLIP puro en la tarea zero-shot para cultivos de Banana).

### 4.2 Métricas del Recuperador (ChromaDB K-NN Quality)
Para evaluar qué tan bien está funcionando la fusión temprana de embeddings (visual + texto) al recuperar conocimiento pertinente de ChromaDB, se analizan los $K$ vecinos más cercanos ($K=3$) mediante tres indicadores de vanguardia en sistemas de recuperación de información (IR):
* **Recall@K:** Mide si la clase real de la planta se encuentra presente en al menos uno de los tres documentos recuperados de la base de datos de conocimiento.
* **Precision@K (Pureza):** Evalúa la proporción de los vecinos recuperados que pertenecen exactamente a la misma clase fitopatológica que la muestra de prueba.
* **MRR (Mean Reciprocal Rank):** Penaliza matemáticamente al recuperador basándose en la posición del primer acierto. Si el vecino correcto aparece en la primera posición, el puntaje es $1.0$; si aparece en la segunda, es $0.5$, y así sucesivamente.

### 4.3 Métricas del Generador (Calidad Teórica del Texto)
Finalmente, se evalúa la capacidad del LLM para sintetizar descripciones diagnósticas concisas y alineadas con los síntomas morfológicos reales de la hoja, comparando de forma cruzada la justificación final (`justificacion_final_llm`) contra el subtítulo ciego original (`descripcion_sintomas_censurada`):
* **Token F1-Score:** Mide la coincidencia y superposición armónica palabra por palabra a nivel de tokens individuales.
* **ROUGE-L:** Evalúa la co-ocurrencia de n-gramas basándose en la Máxima Secuencia Común (LCS), lo que determina si el orden y la estructura de la explicación sintomática guardan coherencia con la referencia.
* **BERTScore (Similitud Semántica Profunda):** A diferencia de las métricas de coincidencia exacta de palabras, BERTScore utiliza un modelo de lenguaje basado en Transformers (`distilbert-base-uncased`) para computar la similitud de los embeddings de los tokens. Esto permite evaluar la calidad del texto a nivel conceptual y semántico, reconociendo sinónimos y paráfrasis médicas correctas aunque no usen las mismas palabras textuales.

Todos los resultados analíticos se consolidan automáticamente en una tabla comparativa y se exportan al archivo físico `/content/reporte_metricas_fase4.csv` para su posterior documentación y graficación en el reporte final del proyecto.

In [11]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

def calcular_token_f1(pred_text, ref_text):
    """Calcula el F1-Score a nivel de tokens individuales (palabras)."""
    pred_tokens = str(pred_text).lower().split()
    ref_tokens = str(ref_text).lower().split()

    if not pred_tokens or not ref_tokens:
        return 0.0

    common = set(pred_tokens) & set(ref_tokens)
    num_same = sum(min(pred_tokens.count(w), ref_tokens.count(w)) for w in common)

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(ref_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * (precision * recall) / (precision + recall)

def main():
    print("=== Iniciando Fase 4: Evaluacion de Metricas del Sistema RAG ===")

    # Verificación de la ruta y existencia del archivo de resultados de la Fase 3
    ruta_resultados = "/content/resultados_canalizacion_rag.csv"
    if not os.path.exists(ruta_resultados):
        print(f"Error: No se encontro el archivo de resultados en {ruta_resultados}")
        return

    df = pd.read_csv(ruta_resultados)
    total_muestras = len(df)
    print(f"Registros cargados para evaluacion: {total_muestras}")

    # Bloque 1: Cálculo del Accuracy global de clasificación vs la etiqueta real
    y_true = df['folder_real'].astype(str).str.strip().str.lower()
    y_pred = df['prediccion_llm'].astype(str).str.strip().str.lower()
    accuracy_final = accuracy_score(y_true, y_pred) * 100

    # Bloque 2: Evaluación del rendimiento y pureza del recuperador (ChromaDB K-NN)
    recalls_at_k = []
    precisions_at_k = []
    rr_list = []

    # Procesamiento por registro para extraer y evaluar los 3 vecinos recuperados
    for idx, row in df.iterrows():
        clase_real = str(row['folder_real']).strip().lower()

        metadata_vecinos = [
            str(row['metadata_vecino_1']).strip().lower(),
            str(row['metadata_vecino_2']).strip().lower(),
            str(row['metadata_vecino_3']).strip().lower()
        ]

        # Un HIT real (1) ocurre si la clase evaluada está contenida en el metadato objetivo del fragmento recuperado
        hits = [1 if clase_real in v else 0 for v in metadata_vecinos]

        # Cálculo secuencial de Recall@k, Precision@k y Mean Reciprocal Rank (MRR)
        recalls_at_k.append(1 if sum(hits) > 0 else 0)
        precisions_at_k.append(sum(hits) / len(hits))

        rank_pos = 0
        for pos, hit in enumerate(hits):
            if hit == 1:
                rank_pos = pos + 1
                break
        mrr_score = 1.0 / rank_pos if rank_pos > 0 else 0.0
        rr_list.append(mrr_score)

    recall_k_final = np.mean(recalls_at_k) * 100
    precision_k_final = np.mean(precisions_at_k) * 100
    mrr_final = np.mean(rr_list)

    # Bloque 3: Evaluación n-grama y semántica del texto generado por el LLM
    token_f1_list = []
    rouge_l_list = []
    scorer_rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

    referencias = df['descripcion_sintomas_censurada'].astype(str).tolist()
    generaciones = df['justificacion_final_llm'].astype(str).tolist()

    # Comparación de la justificación generada contra el subtítulo base (Token F1 y ROUGE-L)
    for gen, ref in zip(generaciones, referencias):
        token_f1_list.append(calcular_token_f1(gen, ref))
        scores_rouge = scorer_rouge.score(ref, gen)
        rouge_l_list.append(scores_rouge['rougeL'].fmeasure)

    # Extracción de similitud semántica contextual profunda mediante BERTScore
    print("Calculando BERTScore (esto puede tomar unos segundos en GPU)...")
    try:
        P_bert, R_bert, F1_bert = bert_score_fn(
            generaciones,
            referencias,
            lang="en",
            model_type="distilbert-base-uncased",
            verbose=False
        )
        bert_score_final = F1_bert.mean().item()
    except Exception as e:
        print(f"Advertencia al calcular BERTScore: {e}")
        bert_score_final = 0.0

    token_f1_final = np.mean(token_f1_list)
    rouge_l_final = np.mean(rouge_l_list)

    # Impresión estructurada en consola y exportación del DataFrame de métricas a CSV
    print("\n" + "="*50)
    print("TABLA DE METRICAS CONFIGURADA - FASE 4")
    print("="*50)

    resultados_data = [
        ["Accuracy (Clasificacion vs Baseline)", f"{accuracy_final:.2f}%"],
        ["Recall@k (Encontrado)", f"{recall_k_final:.2f}%"],
        ["Precision@k (Pureza)", f"{precision_k_final:.2f}%"],
        ["MRR (Posicion)", f"{mrr_final:.4f}"],
        ["Token F1 (Generacion)", f"{token_f1_final:.4f}"],
        ["ROUGE-L (Generacion)", f"{rouge_l_final:.4f}"],
        ["BERTScore (Semantica)", f"{bert_score_final:.4f}"]
    ]

    metrics_df = pd.DataFrame(resultados_data, columns=["Metrica", "Valor Obtenido"])
    print(metrics_df.to_string(index=False))
    print("="*50)

    metrics_df.to_csv("/content/reporte_metricas_fase4.csv", index=False, encoding='utf-8')
    print("Los datos analiticos de la evaluacion se guardaron en: /content/reporte_metricas_fase4.csv")

if __name__ == "__main__":
    main()

=== Iniciando Fase 4: Evaluacion de Metricas del Sistema RAG ===
Registros cargados para evaluacion: 90
Calculando BERTScore (esto puede tomar unos segundos en GPU)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



TABLA DE METRICAS CONFIGURADA - FASE 4
                             Metrica Valor Obtenido
Accuracy (Clasificacion vs Baseline)         11.11%
               Recall@k (Encontrado)          7.78%
                Precision@k (Pureza)          7.78%
                      MRR (Posicion)         0.0778
               Token F1 (Generacion)         0.1535
                ROUGE-L (Generacion)         0.1581
               BERTScore (Semantica)         0.7445
Los datos analiticos de la evaluacion se guardaron en: /content/reporte_metricas_fase4.csv
